In [ ]:
# === Cell 1: Paste your FrankenMSA link here (with auto-clean) ===
#@title Paste your FrankenMSA link here
#@markdown Paste the full URL copied from the FrankenMSA web page below, then run this cell.
#@markdown The script will automatically parse the parameters and download or upload your PDB file.

# --- Auto-clean workspace for a fresh start ---
import os, gc
from IPython.display import clear_output

print("🧹 Cleaning up old files...")
os.system("rm -rf /content/ProteinMPNN/outputs_run/* 2>/dev/null")
os.system("find /content -maxdepth 2 -type f -name '*.pdb' -delete 2>/dev/null")
os.system("rm -f /content/*.zip /content/*.fa /content/*.fasta /content/*.a3m 2>/dev/null")
gc.collect()
clear_output(wait=True)
print("✅ Workspace cleaned. Ready for a fresh run!\n")


# --- Parameter parsing section ---
from urllib.parse import urlparse, parse_qs
import json, re
from google.colab import files

link = "?temp=2.3&num=130&design=&fixed=&pdb=2NNC&homomer=1"  #@param {type:"string"}

# --- move the detailed instruction block BELOW the input box ---
#@markdown ---
#@markdown ### 🔗 Instructions
#@markdown 1. **Paste the full URL** copied from the FrankenMSA web page into the box above.
#@markdown 2. If your link includes a valid **PDB code**, the script will **automatically download** the structure file from the RCSB database.
#@markdown 3. If the **PDB field is empty**, please **upload your own `.pdb` file** manually:
#@markdown    - After running this cell, click the **“Choose File”** button that appears below to upload your `.pdb`.
#@markdown 4. Once the PDB file is ready, run all remaining cells in order (**Runtime → Run all**) to execute the ProteinMPNN workflow.
#@markdown 5. When finished, a **ZIP file containing all results** will be automatically generated and ready for download.

# If empty, ask for manual input
if not link.strip():
    link = input("Please paste the full FrankenMSA Colab link:\n").strip()

# Extract query parameters
parsed = urlparse(link)
q = parsed.query if parsed.query else link.lstrip('?#')  
params = {k: v[0] for k, v in parse_qs(q).items()}

def _get_float(k, d):
    try: return float(params.get(k, d))
    except: return d

def _get_int(k, d):
    try: return int(params.get(k, d))
    except: return d

def _norm_csv(k):
    return (params.get(k, "") or "").replace(" ", "").upper()

sampling_temp = _get_float("temp", 1.0)
num_seqs = _get_int("num", 128)
pdb_code = _norm_csv("pdb")
homomer = (params.get("homomer", "1") == "1")
design = _norm_csv("design")
fixed = _norm_csv("fixed")

print("✅ Parameters successfully parsed:")
print(json.dumps({
    "sampling_temp": sampling_temp,
    "num_seqs": num_seqs,
    "pdb_code": pdb_code,
    "homomer": homomer,
    "design": design,
    "fixed": fixed
}, indent=2))

# --- Download or upload PDB ---
import os, time

def get_pdb(pdb_code):
    from google.colab import files  
    if not pdb_code:
        print("📤 No PDB code provided. Please upload your local .pdb file:")
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {filename}")
        return filename

    code = str(pdb_code).strip().upper()
    filename = f"{code}.pdb"

    # RCSB(view) -> RCSB(download) -> PDBe
    urls = [
        f"https://files.rcsb.org/view/{code}.pdb",
        f"https://files.rcsb.org/download/{code}.pdb",
        f"https://www.ebi.ac.uk/pdbe/entry-files/download/{code}.pdb",
    ]

    def _ok(fp):
        return os.path.exists(fp) and os.path.getsize(fp) > 0

    success = False
    for url in urls:
        print(f"🌐 Trying to download from {url} ...")
        for attempt in range(1, 5):  
            cmd = f"wget -q -O '{filename}' --tries=3 --timeout=30 --no-check-certificate -L '{url}'"
            rc = os.system(cmd)
            if rc == 0 and _ok(filename):
                print(f"✅ Downloaded: {filename}")
                success = True
                break
            else:
                print(f"⚠️ Attempt {attempt} failed, retrying...")
                time.sleep(min(2 * attempt, 8))  
        if success:
            break

    if not success:
        raise RuntimeError(
            f"❌ Failed to download {filename}. "
            f"Please check the PDB code or try uploading manually."
        )
    return filename

pdb_path = get_pdb(pdb_code)

In [ ]:
# === Cell 2: Setup ProteinMPNN environment (GPU, repo, deps, weights) ===
import os, sys, subprocess, json, shutil

# 1) GPU check (Colab should have torch preinstalled)
try:
    import torch
    has_cuda = torch.cuda.is_available()
    print(f"CUDA available: {has_cuda}")
    if has_cuda:
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("⚠️ PyTorch not found or CUDA check failed:", e)

# 2) Clone ProteinMPNN repo if needed
ROOT = "/content/ProteinMPNN"
if not os.path.isdir(ROOT):
    print("📥 Cloning ProteinMPNN...")
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/dauparas/ProteinMPNN.git", ROOT],
        check=True
    )
else:
    print("✅ ProteinMPNN repo already present.")
sys.path.append(ROOT)

# 3) Minimal Python deps (quiet install)
print("📦 Installing Python deps (quiet)...")
# biopython + einops are typically enough for the quick demo
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "biopython==1.83", "einops==0.7.0"],
    check=False,
)

# 4) Model/weights flags (can be changed later if you expose switches in Cell 1)
model_name = globals().get("model_name", "v_48_020")        # v_48_002 / v_48_010 / v_48_020 / v_48_030
use_soluble_model = bool(globals().get("use_soluble_model", False))
ca_only = bool(globals().get("ca_only", False))

def _pick_weights_root(root, ca=False, soluble=False):
    if ca:
        return os.path.join(root, "ca_model_weights")
    if soluble:
        return os.path.join(root, "soluble_model_weights")
    return os.path.join(root, "vanilla_model_weights")

WEIGHTS_ROOT = _pick_weights_root(ROOT, ca_only, use_soluble_model)

# 5) Check weights presence (we don't force-download to keep the cell stable)
def _weights_ok(folder: str) -> bool:
    if not os.path.isdir(folder):
        return False
    try:
        # any .pt file inside the folder is a good sign
        for f in os.listdir(folder):
            if f.endswith(".pt"):
                return True
    except Exception:
        pass
    return False

weights_ready = _weights_ok(WEIGHTS_ROOT)
if not weights_ready:
    print("⚠️ Model weights not found at:", WEIGHTS_ROOT)
    print("   Please download the official weights as per the ProteinMPNN README,")
    print("   and place them under the corresponding folder. You can still proceed,")
    print("   but the run cell will fail until weights are available.")

# 6) Output directory
OUT_DIR = os.path.join(ROOT, "outputs_run")
os.makedirs(OUT_DIR, exist_ok=True)

# 7) Summary
print("\n=== Environment summary ===")
print(json.dumps({
    "repo_root": ROOT,
    "model_name": model_name,
    "use_soluble_model": use_soluble_model,
    "ca_only": ca_only,
    "weights_root": WEIGHTS_ROOT,
    "weights_ready": weights_ready,
    "out_dir": OUT_DIR,
    "cuda_available": bool('torch' in globals() and getattr(torch, 'cuda', None) and torch.cuda.is_available())
}, indent=2))
print("===========================\n")

In [ ]:
# === Cell 3: Run ProteinMPNN and collect outputs (FASTA + minimal A3M) ===
import os, sys, json, glob, subprocess
from pathlib import Path

# --- Safety checks from previous cells ---
assert 'pdb_path' in globals() and os.path.isfile(pdb_path), "pdb_path missing - please run Cell 1 (PDB) first."
assert 'ROOT' in globals() and 'OUT_DIR' in globals() and 'WEIGHTS_ROOT' in globals(), "Run Cell 2 (env setup) first."

# Inputs with defaults (in case user re-runs cells out of order)
model_name           = globals().get("model_name", "v_48_020")
use_soluble_model    = bool(globals().get("use_soluble_model", False))
ca_only              = bool(globals().get("ca_only", False))
sampling_temp        = str(globals().get("sampling_temp", "0.1"))
num_seqs             = int(globals().get("num_seqs", 128))
designed_chain_list  = list(globals().get("designed_chain_list", []))
fixed_chain_list     = list(globals().get("fixed_chain_list", []))

# Prepare optional chain design spec JSONL (only if user specified something)
chain_jsonl = None
if designed_chain_list or fixed_chain_list:
    chain_jsonl = os.path.join(OUT_DIR, "chain_id.jsonl")
    obj = {
        "name": Path(pdb_path).stem,
        "design_chain_list": designed_chain_list,
        "fixed_chain_list": fixed_chain_list,
    }
    with open(chain_jsonl, "w") as f:
        f.write(json.dumps(obj) + "\n")

# Build command
cmd = [
    sys.executable, f"{ROOT}/protein_mpnn_run.py",
    "--pdb_path", pdb_path,
    "--out_folder", OUT_DIR,
    "--model_name", model_name,
    "--path_to_model_weights", WEIGHTS_ROOT,
    "--num_seq_per_target", str(num_seqs),
    "--sampling_temp", sampling_temp,
]

if use_soluble_model:
    cmd.append("--use_soluble_model")
if ca_only:
    cmd.append("--ca_only")
if chain_jsonl:
    cmd.extend(["--chain_id_jsonl", chain_jsonl])

print("🔧 Command:\n", " ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print("=== STDOUT ===\n", proc.stdout[:2000])
print("\n=== STDERR ===\n", proc.stderr[:2000])

if proc.returncode != 0:
    raise RuntimeError("ProteinMPNN run failed. See logs above.")

# Collect outputs into single FASTA (fast + early stop)
from pathlib import Path
import time

stem = Path(pdb_path).stem
fasta_out = os.path.join(OUT_DIR, f"{stem}_proteinmpnn.fasta")
a3m_out   = os.path.join(OUT_DIR, f"{stem}_proteinmpnn.a3m")

# 仅收集与本次目标 PDB 相关的结果，避免把历史 run 混进来
cands = []
for fn in glob.glob(os.path.join(OUT_DIR, "**", "*.fa*"), recursive=True):
    try:
        p = Path(fn)
        if stem in p.stem or stem in str(p.parent):
            cands.append(p)
    except Exception:
        pass
cands = sorted(cands)

print(f"\n🔍 Collecting from {len(cands)} files for target '{stem}'")

def _merge_fast(files, max_seqs):
    i = 0
    t0 = time.time()
    # 大缓冲写入，减少 I/O 调用次数
    with open(fasta_out, "w", buffering=1 << 20) as fout_fa, \
         open(a3m_out,   "w", buffering=1 << 20) as fout_a3m:
        for fn in files:
            with open(fn, "r") as fin:
                for line in fin:
                    if not line:
                        continue
                    if line[0] == ">":
                        i += 1
                        fout_fa.write(f">sample_{i}\n")
                        fout_a3m.write(f">sample_{i}\n")
                        if i % 50 == 0:
                            print(f"  ...merged {i} seqs")
                        if i >= max_seqs:   # 达到目标数量提前停止
                            print(f"⏱️ Merge time: {time.time()-t0:.1f}s (early stop at {i})")
                            return i
                    else:
                        # 仅去掉末尾换行，更快；再写回 '\n'
                        if line.endswith("\n"):
                            seq = line[:-1]
                        else:
                            seq = line
                        fout_fa.write(seq + "\n")
                        fout_a3m.write(seq + "\n")
    print(f"⏱️ Merge time: {time.time()-t0:.1f}s (total {i})")
    return i

merged = _merge_fast(cands, max_seqs=num_seqs)
print(f"✅ Merged FASTA: {fasta_out} (N={merged})")
print(f"✅ Minimal A3M : {a3m_out}")

# Offer downloads in Colab
try:
    from google.colab import files as colab_files
    print("⬇️  Use the links below to download:")
    print(f"FASTA: {fasta_out}")
    print(f"A3M  : {a3m_out}")
    # 若要自动弹出下载，可取消下行注释：
    # colab_files.download(fasta_out)
    # colab_files.download(a3m_out)
except Exception:
    pass


In [ ]:
# === Cell 4: Package and download output files ===
import os, zipfile
from pathlib import Path

assert 'OUT_DIR' in globals(), "Please run Cell 2 and Cell 3 first."
stem = Path(pdb_path).stem if 'pdb_path' in globals() else 'proteinmpnn_output'

# Locate expected outputs
fasta_out = os.path.join(OUT_DIR, f"{stem}_proteinmpnn.fasta")
a3m_out   = os.path.join(OUT_DIR, f"{stem}_proteinmpnn.a3m")

missing = []
for f in [fasta_out, a3m_out]:
    if not os.path.isfile(f):
        missing.append(f)
if missing:
    print("⚠️ Missing expected outputs:", missing)
else:
    print("✅ Found outputs:")
    print("FASTA:", fasta_out)
    print("A3M  :", a3m_out)

# Create ZIP package
zip_path = os.path.join(OUT_DIR, f"{stem}_proteinmpnn_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if os.path.isfile(fasta_out): zf.write(fasta_out, os.path.basename(fasta_out))
    if os.path.isfile(a3m_out):   zf.write(a3m_out,   os.path.basename(a3m_out))

print(f"📦 Created ZIP archive: {zip_path}")

# Offer download (Colab only)
try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
    print("⬇️ Download should start automatically.")
except Exception as e:
    print("⚠️ Automatic download unavailable:", e)
    print(f"Manual download path: {zip_path}")